# 🔬 PATH A1: Merge QLoRA Adapter → Xuất GGUF (Q4_K_M) → Upload HF Hub

- **LoRA Adapter:** `hung2903/gemma-4-E4B-unsloth-vaccine-xai`
- **Base Model:** `unsloth/gemma-4-E4B-it`
- **Output Repo:** `hung2903/gemma-4-E4B-vaccine-xai-merged`
- **Mục tiêu:** Xuất file GGUF ~2.5–3GB để chạy Local với LM Studio / Ollama / llama.cpp

> ⚠️ **Kaggle Disk Limit: 20GB.** Notebook này được tối ưu để tránh lỗi `OSError: No space left on device`.
> Chiến lược: Xóa HF cache (~5GB) ngay TRƯỚC khi chạy `push_to_hub_gguf` để giải phóng đủ không gian.

In [ ]:
%%capture
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps "trl<0.9.0" peft accelerate bitsandbytes
!pip install -q huggingface_hub

In [ ]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"GPU count: {torch.cuda.device_count()}")

from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
HF_TOKEN = secrets.get_secret("HF_TOKEN")

login(token=HF_TOKEN)
print("✅ Logged in to HuggingFace Hub")

In [ ]:
LORA_ADAPTER = "hung2903/gemma-4-E4B-unsloth-vaccine-xai"
UNSLOTH_BASE = "unsloth/gemma-4-E4B-it"
MERGED_REPO = "hung2903/gemma-4-E4B-vaccine-xai-merged"
MAX_SEQ_LENGTH = 2048
GGUF_METHOD = "q4_k_m"

In [ ]:
# ✅ KHÔNG gọi FastModel.for_inference() ở đây.
# for_inference() patch internal state và không tương thích với merge — gây corrupt.
from unsloth import FastModel
from unsloth.chat_templates import get_chat_template

print(f"Loading adapter: {LORA_ADAPTER}")
print(f"Base: {UNSLOTH_BASE}")
print("⏳ This will download ~50MB adapter + ~5GB base...")

model, tokenizer = FastModel.from_pretrained(
    model_name=LORA_ADAPTER,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
    token=HF_TOKEN,
)
tokenizer = get_chat_template(tokenizer, chat_template="gemma-4")

print(f"\n✅ Loaded successfully")
print(f"Model class: {type(model).__name__}")
print(f"GPU memory used: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

In [ ]:
# ✅ Kiểm tra output của model trước khi merge
# Dùng model.eval() thủ công — KHÔNG dùng FastModel.for_inference()
test_prompt = """Bạn là Trí tuệ Nhân tạo có khả năng giải thích (Explainable AI) trong lĩnh vực Y tế Công cộng.
Hãy phân tích văn bản sau đây về chủ đề vắc-xin, đưa ra lý luận chi tiết HOÀN TOÀN bằng tiếng Việt
về tính xác thực, thái độ và cảm xúc. Tuyệt đối không dùng tiếng Anh.

Văn bản: Vắc-xin COVID gây vô sinh ở phụ nữ trẻ và biến đổi gen ở trẻ em."""

messages = [{
    "role": "user",
    "content": [{"type": "text", "text": test_prompt}]
}]

# Bước 1: Format thành string, KHÔNG tokenize ngay
formatted_text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)

# Bước 2: Tokenize thủ công với keyword text= tường minh
# Gemma4Processor: signature (self, images, text, ...) → PHẢI dùng text=
_device = next(model.parameters()).device
model_inputs = tokenizer(text=formatted_text, return_tensors="pt").to(_device)

print("Generating test response...")
model.eval()  # Set eval mode thủ công, KHÔNG dùng for_inference()
with torch.no_grad():
    outputs = model.generate(
        input_ids=model_inputs["input_ids"],
        attention_mask=model_inputs["attention_mask"],
        max_new_tokens=300,
        temperature=0.7,
        do_sample=True,
        use_cache=True,
    )
response = tokenizer.decode(outputs[0][model_inputs["input_ids"].shape[-1]:], skip_special_tokens=True)
print("\n=== TEST OUTPUT ===")
print(response)

## 🧹 Bước quan trọng: Dọn cache trước khi xuất GGUF

**Nguyên nhân lỗi `OSError: No space left on device`:**
- Kaggle giới hạn `/kaggle/working` chỉ **20GB**.
- Sau khi tải base model (~5GB vào cache) và chạy inference, còn lại ~8-9GB.
- `save_pretrained_gguf` cần: merge 16-bit (~8GB tạm thời) + clone llama.cpp (~2GB) + file GGUF (~3GB) = **~13GB** → Tràn!

**Giải pháp:** Xóa HF cache (~5GB) NGAY TRƯỚC khi gọi `push_to_hub_gguf` để giải phóng không gian.

In [ ]:
# ✅ BƯỚC PRE-CLEAN: Giải phóng HF cache trước khi xuất GGUF
import shutil
import os
import subprocess

print("=" * 50)
print("KIỂM TRA DISK TRƯỚC KHI XUẤT GGUF")
print("=" * 50)

# Hiển thị trạng thái disk
df_before = subprocess.check_output(["df", "-h", "/kaggle/working"]).decode()
print(f"Disk status (trước khi dọn):\n{df_before}")

# Xoá HF hub cache để giải phóng ~5GB
hf_cache = "/root/.cache/huggingface/hub"
if os.path.exists(hf_cache):
    shutil.rmtree(hf_cache)
    print(f"✅ Đã xóa HF cache: {hf_cache}")
else:
    print("ℹ️ HF cache không tồn tại, bỏ qua.")

df_after = subprocess.check_output(["df", "-h", "/kaggle/working"]).decode()
print(f"Disk status (sau khi dọn):\n{df_after}")
print("\n✅ Sẵn sàng xuất GGUF!")

## 🏭 Xuất GGUF (Q4_K_M) — Nén mô hình xuống ~2.5–3GB

Sử dụng `push_to_hub_gguf` của Unsloth — hàm này:
1. Merge LoRA weights vào base model (16-bit tạm thời)
2. Chạy llama.cpp để lượng tử hóa sang Q4_K_M (~3GB)
3. Push file `.gguf` trực tiếp lên HF Hub

> ⏰ Quá trình mất khoảng **15–25 phút** trên Kaggle T4.

In [ ]:
# ✅ XUẤT VÀ ĐẨY ĐỊNH DẠNG GGUF LÊN HUGGING FACE HUB
# Bước model.train() là BẮT BUỘC: reset model về training state
# trước khi merge để dequantize và hợp nhất weights đúng cách.
print("Chuẩn bị merge weights...")
model.train()

print(f"Đang đẩy file GGUF ({GGUF_METHOD}) lên Hugging Face Hub: {MERGED_REPO}")
print("⏳ Quá trình này mất 15–25 phút (merge + llama.cpp build + quantize + upload)...")

model.push_to_hub_gguf(
    MERGED_REPO,
    tokenizer,
    quantization_method=GGUF_METHOD,
    token=HF_TOKEN,
)

print("\n✅ Đã tối ưu hóa Notebook để xuất mô hình GGUF cho Local Inference")
print(f"🔗 https://huggingface.co/{MERGED_REPO}")

In [ ]:
# ✅ KIỂM TRA DUNG LƯỢNG FILE GGUF VỪA XUẤT
# File GGUF Q4_K_M của Gemma-4 4B đạt chuẩn khi nằm trong khoảng 2.5–3.5 GB
import os
import glob

gguf_files = glob.glob("/kaggle/working/**/*.gguf", recursive=True)
gguf_files += glob.glob("/kaggle/working/*.gguf")

if gguf_files:
    print("=" * 50)
    print("KẾT QUẢ KIỂM TRA FILE GGUF")
    print("=" * 50)
    for f in gguf_files:
        size_bytes = os.path.getsize(f)
        size_gb = size_bytes / (1024 ** 3)
        status = "✅ ĐẠT CHUẨN" if 2.0 <= size_gb <= 4.0 else "⚠️ NGOÀI NGƯỠNG (kiểm tra lại)"
        print(f"  File : {os.path.basename(f)}")
        print(f"  Dung lượng: {size_gb:.2f} GB ({size_bytes:,} bytes)")
        print(f"  Đánh giá  : {status}")
        print()
else:
    print("⚠️ Không tìm thấy file .gguf trong /kaggle/working.")
    print("   push_to_hub_gguf có thể đã upload trực tiếp mà không lưu local.")
    print("   Kiểm tra repo trên HF Hub để xác nhận.")

In [ ]:
# ✅ Cleanup cuối: Dọn dẹp disk sau khi upload thành công
import shutil
import os
import subprocess

print("Freeing remaining disk space...")

# Xóa HF hub cache (nếu còn)
hf_cache = "/root/.cache/huggingface/hub"
if os.path.exists(hf_cache):
    shutil.rmtree(hf_cache)
    print(f"✅ Deleted {hf_cache}")

# Xóa thư mục GGUF local (nếu tồn tại)
for path in glob.glob("/kaggle/working/vaccinenlp_gemma4_gguf*"):
    if os.path.isdir(path):
        shutil.rmtree(path)
    else:
        os.remove(path)
    print(f"✅ Deleted {path}")

# Hiển thị disk còn lại
df = subprocess.check_output(["df", "-h", "/kaggle/working"]).decode()
print(f"\nDisk status sau cleanup:\n{df}")

In [ ]:
# ✅ Upload Model Card lên HF Hub
from huggingface_hub import HfApi
from io import BytesIO

api = HfApi(token=HF_TOKEN)

MODEL_CARD = """---
license: gemma
base_model: unsloth/gemma-4-E4B-it
tags:
  - vietnamese
  - vaccine
  - misinformation
  - public-health
  - explainable-ai
  - gguf
  - q4_k_m
language:
  - vi
pipeline_tag: text-generation
inference: true
---

# VaccineNLP — Gemma-4 E4B Reasoning Engine (GGUF Q4_K_M)

File GGUF (Q4_K_M, ~3GB) của [hung2903/gemma-4-E4B-unsloth-vaccine-xai](https://huggingface.co/hung2903/gemma-4-E4B-unsloth-vaccine-xai)
QLoRA adapter merged với base [unsloth/gemma-4-E4B-it](https://huggingface.co/unsloth/gemma-4-E4B-it).

## Mục đích

**XAI Reasoning Engine** cho hệ thống VaccineNLP — giải thích Chain-of-Thought về phát hiện thông tin sai lệch vaccine bằng tiếng Việt.

## Hiệu suất (Gold Test Set, n=186)

| Metric | Score |
|---|:---:|
| Macro F1 Misinfo | 0.6925 |
| Macro F1 Stance | 0.5818 |
| **Macro F1 Sentiment** | **0.7196** |
| Parse Success Rate | 66.7% |

## Sử dụng với LM Studio / Ollama (Local)

```python
# Sau khi nạp file .gguf vào LM Studio và bật Local Server:
import openai
client = openai.OpenAI(base_url="http://localhost:1234/v1", api_key="lm-studio")
response = client.chat.completions.create(
    model="local-model",
    messages=[{"role": "user", "content": "Van ban: Vac-xin COVID gay vo sinh."}],
    max_tokens=512,
    temperature=0.1,
)
print(response.choices[0].message.content)
```

## Trích dẫn

```bibtex
@thesis{vaccinenlp2026,
  title={Ứng dụng Xử lý Ngôn ngữ Tự nhiên trong phát hiện thông tin sai lệch về vaccine},
  author={Kim Mạnh Hưng and Đinh Lê Quỳnh Phương},
  school={Trường Đại học Y tế Công cộng (HUPH)},
  year={2026},
}
```
"""

api.upload_file(
    path_or_fileobj=BytesIO(MODEL_CARD.encode("utf-8")),
    path_in_repo="README.md",
    repo_id=MERGED_REPO,
    repo_type="model",
)
print("✅ Model card uploaded")
print(f"🔗 https://huggingface.co/{MERGED_REPO}")

In [ ]:
# ❌ [ĐÃ VÔ HIỆU HÓA] Test HF Inference API
# Lý do: Định dạng GGUF không tương thích với HF Inference API serverless.
# Việc kiểm thử đầy đủ được thực hiện trực tiếp trên máy cá nhân thông qua LM Studio.

# import requests, time
# API_URL = f"https://api-inference.huggingface.co/models/{MERGED_REPO}"
# headers = {"Authorization": f"Bearer {HF_TOKEN}"}
# response = requests.post(API_URL, headers=headers, json={...})

print("Quy trình xuất GGUF hoàn tất!")
print(f"Tải file .gguf tại: https://huggingface.co/{MERGED_REPO}")
print("Nạp vào LM Studio → Bật Local Server → Kết nối App Gradio tại http://localhost:1234/v1")